## Building A Chatbot
In this Section We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")



In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

d:\Jeyasuriya\Learning\Machine Learning\LLM\GenAI\genai-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.12'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013DFCB470E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013DFCB47E00>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Jeyasuriya and I am a Chief AI Engineer")])

AIMessage(content="Nice to meet you, Jeyasuriya! It's great to have a Chief AI Engineer like yourself interacting with me. What brings you here today? Are you looking to explore new AI developments, discuss a specific project, or perhaps get some insights on AI-related topics? I'm all ears and ready to help.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 52, 'total_tokens': 118, 'completion_time': 0.102576976, 'completion_tokens_details': None, 'prompt_time': 0.003339903, 'prompt_tokens_details': None, 'queue_time': 0.048892986, 'total_time': 0.105916879}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7962-6ed8-7791-805e-fd6599c4a867-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 66, 'total_tokens': 118})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Jeyasuriya and I am a Chief AI Engineer"),
        AIMessage(content="Hello Jeyasuriya! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Jeyasuriya, and you're a Chief AI Engineer. That's a very interesting and in-demand role! \n\nCan you tell me more about what your typical day is like in that position, or what kind of AI-related projects you're currently working on? I'm all ears!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 125, 'total_tokens': 188, 'completion_time': 0.125519653, 'completion_tokens_details': None, 'prompt_time': 0.007974502, 'prompt_tokens_details': None, 'queue_time': 0.057816967, 'total_time': 0.133494155}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7963-5bdf-75a0-9ebb-d6d5d97d9940-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 63, 'total_tokens': 188})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
!pip install langchain_community

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\jayasuriya\AppData\Local\Temp\ipykernel_86364\2274412368.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
d:\Jeyasuriya\Learning\Machine Learning\LLM\GenAI\genai-venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Jeyasuriya and I am a Data Engineer")],
    config=config
)

In [8]:
response.content

"Nice to meet you, Jeyasuriya! As a Data Engineer, you must be working with data to design, build, and maintain the architectures that enable data processing, analysis, and storage. That's a fascinating field with a lot of opportunities for growth and innovation.\n\nWhat specific areas of Data Engineering do you specialize in? Are you more into data warehousing, ETL (Extract, Transform, Load), data pipelines, or perhaps machine learning and AI? Or maybe you're interested in cloud-based data engineering, such as AWS or GCP?"

In [9]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Jeyasuriya.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 178, 'total_tokens': 188, 'completion_time': 0.014279013, 'completion_tokens_details': None, 'prompt_time': 0.013374082, 'prompt_tokens_details': None, 'queue_time': 0.049971218, 'total_time': 0.027653095}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7964-e451-7631-8ec8-3d9ebec1426a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 178, 'output_tokens': 10, 'total_tokens': 188})

In [10]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't have any information about your name. Our conversation has just started, and I don't retain any data about individual users. If you'd like to share your name, I'd be happy to chat with you."

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

'Hi John, nice to meet you. How are you doing today?'

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [13]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [14]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Jeyasuriya")]})

AIMessage(content="Hello Jeyasuriya, it's nice to meet you. Is there something I can help you with today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 60, 'total_tokens': 85, 'completion_time': 0.036505038, 'completion_tokens_details': None, 'prompt_time': 0.00357121, 'prompt_tokens_details': None, 'queue_time': 0.050258696, 'total_time': 0.040076248}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7966-2477-7f90-86c1-63c6d256f952-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 25, 'total_tokens': 85})

In [15]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

d:\Jeyasuriya\Learning\Machine Learning\LLM\GenAI\genai-venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [16]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Jeyasuriya")],
    config=config
)

response

AIMessage(content="Nice to meet you, Jeyasuriya. It's great that you've taken the first step to interact with me. I'm here to assist you with any questions, topics, or problems you'd like to discuss. How can I help you today? Is it a specific topic or something on your mind that you'd like to talk about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 72, 'prompt_tokens': 60, 'total_tokens': 132, 'completion_time': 0.104344579, 'completion_tokens_details': None, 'prompt_time': 0.003089536, 'prompt_tokens_details': None, 'queue_time': 0.0484291, 'total_time': 0.107434115}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f7966-773a-7e03-9dda-1a9523ecf0df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 72, 'total_tokens': 132})

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Jeyasuriya.'

In [18]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [19]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Jeyasuriya")],"language":"Tamil"})
response.content

'வணக்கம் Jeyasuriya. உங்களுக்கு உதவ நான் இங்கு இருக்கிறேன். எந்த செய்தியில் உங்களுக்கு உதவ வேண்டும்?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [20]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

d:\Jeyasuriya\Learning\Machine Learning\LLM\GenAI\genai-venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [22]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Jeyasuriya")],"language":"Tamil"},
    config=config
)
repsonse.content

'வணக்கம் ஜெயசூரியா! நான் உங்களுக்கு உதவ இங்கு இருக்கிறேன். உங்களுக்கு எந்த விதமான உதவியை தேவைப்படுகிறீர்கள்?'

In [23]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Tamil"},
    config=config,
)

In [24]:
response.content

'உங்கள் பெயர் ஜெயசூரியா.'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [36]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [41]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"Unfortunately, I don't have any information about your personal preferences. If you want to tell me, I'd be happy to chat about your favorite ice cream flavors!"

In [42]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked the math problem 2 + 2.'

In [43]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

d:\Jeyasuriya\Learning\Machine Learning\LLM\GenAI\genai-venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [44]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't mention your name earlier. I'm happy to chat with you, but I don't have any information about your name. Would you like to share it with me?"

In [45]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't ask a math problem yet. Our conversation just started, and I'm ready to help you with any questions or math problems you'd like to ask. What's on your mind?"